# Week 3: Intelligence Layer — Win Alignment, Benchmarks & Optimization

This notebook demonstrates the full Week 3 intelligence pipeline:

1. Start with a **weak pipeline** — low distress, few states, few sectors
2. **Benchmark** it against historical NMTC winner patterns
3. **Score alignment** with the `WinProbabilityModel`
4. **Get quantified recommendations** to improve competitiveness
5. **Optimize** the pipeline subset to maximize winner alignment
6. **Compare** before vs. after

> **IMPORTANT — Data Limitation:** Scores in this notebook reflect *alignment with historical winner patterns* (CY2020–2024), **not win probability**. The CDFI Fund does not publish non-winner application data, so a true win probability cannot be computed. See `methodology_disclosure` on any score object.

In [ ]:
import sys
sys.path.insert(0, '..')

from nmtcapp.core.application import Application
from nmtcapp.core.cde import CDEProfile
from nmtcapp.core.pipeline import Pipeline, PipelineProject

## 1. Build a Weak Pipeline

Start with a pipeline that has common problems: concentrated in one state, one sector, and mostly standard-LIC distress.

In [ ]:
cde = CDEProfile.sample()

# Build a weak pipeline: single state, single sector, LIC-only distress
weak_pipeline = Pipeline()
for i in range(8):
    p = PipelineProject(
        project_id=f"WEAK-{i:03d}",
        project_name=f"Illinois Real Estate Project {i+1}",
        qalicb_name=f"IL QALICB {i+1}",
        address=f"{100+i} S Michigan Ave",
        city="Chicago",
        state="IL",  # single state
        sector="real_estate",  # single sector
        project_type="real_estate",
        total_project_cost=8_000_000,
        qei_request=5_000_000,
        qlici_amount=5_000_000,
        expected_jobs_created=8,  # low impact intensity
    )
    weak_pipeline.add(p)

app = Application(cde=cde, requested_allocation=45_000_000)
app.add_pipeline(weak_pipeline)
print(f"Weak pipeline: {len(weak_pipeline)} projects, ${sum(p.qei_request for p in weak_pipeline):,.0f} total QEI")

## 2. Benchmark Against Historical Winners

In [ ]:
bc = app.benchmark()
print(bc.summary())

In [ ]:
# Inspect the tier breakdown
print("Tier summary:", bc.tier_summary)
print(f"\nOverall alignment score: {bc.overall_benchmark_score:.1f}/100")
print(f"\nMethodology: {bc.methodology_disclosure[:200]}...")

## 3. Score Win Alignment

The `WinProbabilityModel` computes a 5-dimension alignment score. **This is NOT a win probability** — it measures how closely the application resembles historical winners.

In [ ]:
score = app.score_win_probability()
print(score.summary())

In [ ]:
# Key numbers
print(f"Composite alignment score: {score.composite_score:.1f}/100")
print(f"Competitive tier:          {score.competitive_tier.upper()}")
print(f"Historical acceptance rate:{score.acceptance_rate_baseline:.1%} (recent 4-round avg)")
print("")
print("Dimensional scores:")
for dim, val in score.dimensional_scores.items():
    bar = '█' * int(val / 10) + '░' * (10 - int(val / 10))
    print(f"  {dim.replace('_', ' ').title():<30} {val:5.1f}  [{bar}]")

## 4. Get Quantified Recommendations

In [ ]:
recs = app.recommendations()
print(recs.summary())

In [ ]:
# Quick access to the critical items
critical = [r for r in recs.recommendations if r.priority == 'critical']
print(f"Critical recommendations: {len(critical)}")
for r in critical:
    print(f"  [{r.category.upper()}] {r.finding[:80]}")
    print(f"    → {r.action[:100]}")
    print(f"    Estimate: {r.quantified_improvement}")
    print()

## 5. Pattern Analysis — What Do Winners Look Like?

Before optimizing, let's understand the winner benchmark data.

In [ ]:
from nmtcapp.intelligence.pattern_analysis import analyze_winning_patterns, compare_to_winners
from nmtcapp.data.historical_awards import get_historical_winners

# Historical round data
df = get_historical_winners()
print("CDFI Fund NMTC Allocation Rounds (CY2020–2024):")
print(df[['round', 'applications', 'awards', 'acceptance_rate', 'avg_award']].to_string(index=False))

In [ ]:
patterns = analyze_winning_patterns()
print("Historical winner distress patterns:")
d = patterns['distress']
print(f"  Deep/severe: p25={d['p25_pct_deep_or_severe']:.0%}  p50={d['p50_pct_deep_or_severe']:.0%}  p75={d['p75_pct_deep_or_severe']:.0%}")
print(f"\nHistorical winner geographic patterns:")
g = patterns['geographic']
print(f"  States:  mean={g['mean_states']:.1f}  p50={g['p50_states']:.0f}  p75={g['p75_states']:.0f}")
print(f"  HHI:     mean={g['mean_hhi']:.0f}")
print(f"\nHistorical winner impact benchmarks:")
i = patterns['impact']
print(f"  Jobs/$MM: p25={i['p25_jobs_per_mm_qei']:.0f}  p50={i['p50_jobs_per_mm_qei']:.0f}  p75={i['p75_jobs_per_mm_qei']:.0f}")

In [ ]:
# Where does our pipeline stand vs. winners?
analysis = app.analyze()
comparison = compare_to_winners(analysis.pipeline_result)

print("Gap to winner medians:")
print(f"  Distress:  observed={comparison['distress']['observed_pct_deep_or_severe']:.0%}  "
      f"winner_p50={comparison['distress']['winner_p50']:.0%}  "
      f"label={comparison['distress']['gap_label']}")
print(f"  States:    observed={comparison['geographic']['observed_states']}  "
      f"winner_p50={comparison['geographic']['winner_p50_states']}  "
      f"gap={comparison['geographic']['gap_to_winner_median_states']}")
print(f"  Jobs/$MM:  observed={comparison['impact']['observed_jobs_per_mm_qei']:.1f}  "
      f"winner_p50={comparison['impact']['winner_p50']:.0f}  "
      f"label={comparison['impact']['gap_label']}")

## 6. Build an Improved Pipeline

Based on the recommendations, build a stronger pipeline:

In [ ]:
# Improved pipeline: 8 states, 5 sectors, higher job intensity
states = ["IL", "OH", "TX", "GA", "NY", "MI", "PA", "NC"]
sectors = ["healthcare", "education", "small_business", "community_facility",
           "affordable_housing", "mixed_use", "healthcare", "education"]
jobs = [35, 28, 45, 20, 15, 18, 32, 25]

improved_pipeline = Pipeline()
for i in range(8):
    p = PipelineProject(
        project_id=f"IMPR-{i:03d}",
        project_name=f"{sectors[i].replace('_', ' ').title()} Project — {states[i]}",
        qalicb_name=f"{states[i]} QALICB {i+1}",
        address=f"{100+i} Main St",
        city=["Chicago", "Columbus", "Houston", "Atlanta", "Buffalo",
              "Detroit", "Philadelphia", "Charlotte"][i],
        state=states[i],
        sector=sectors[i],
        project_type="real_estate" if sectors[i] in ("healthcare", "education", "community_facility") else "operating_business",
        total_project_cost=8_000_000,
        qei_request=5_000_000,
        qlici_amount=5_000_000,
        expected_jobs_created=jobs[i],
    )
    improved_pipeline.add(p)

# Add some additional projects in rural areas
for i, state in enumerate(["KY", "AR", "WV", "MS"]):
    p = PipelineProject(
        project_id=f"RURAL-{i:03d}",
        project_name=f"Rural Healthcare — {state}",
        qalicb_name=f"{state} Rural QALICB",
        address="Rural Route 1",
        city="Rural Community",
        state=state,
        sector="healthcare",
        project_type="real_estate",
        total_project_cost=4_000_000,
        qei_request=2_500_000,
        qlici_amount=2_500_000,
        expected_jobs_created=22,
    )
    improved_pipeline.add(p)

app2 = Application(cde=cde, requested_allocation=55_000_000)
app2.add_pipeline(improved_pipeline)
print(f"Improved pipeline: {len(improved_pipeline)} projects in {len({p.state for p in improved_pipeline})} states")

In [ ]:
# Score the improved pipeline
score2 = app2.score_win_probability()
print(f"BEFORE: {score.composite_score:.1f}/100 [{score.competitive_tier}]")
print(f"AFTER:  {score2.composite_score:.1f}/100 [{score2.competitive_tier}]")
print(f"DELTA:  {score2.composite_score - score.composite_score:+.1f} points")
print()
print("Dimensional comparison:")
for dim in score.dimensional_scores:
    b = score.dimensional_scores[dim]
    a = score2.dimensional_scores[dim]
    print(f"  {dim.replace('_', ' ').title():<30} {b:5.1f} → {a:5.1f}  ({a-b:+.1f})")

## 7. Pipeline Optimizer

The optimizer selects a subset of projects from the pipeline to maximize alignment with historical winners, subject to QEI budget and diversity constraints.

In [ ]:
from nmtcapp.optimizer import OptimizationConstraints, PipelineOptimizer

# Use the improved pipeline and optimize to $45M budget with at least 6 states
constraints = OptimizationConstraints(
    min_total_qei=35_000_000,
    max_total_qei=50_000_000,
    min_projects=8,
    min_states=6,
    required_sectors=["healthcare"],
)

result = PipelineOptimizer(max_iterations=200).optimize(
    improved_pipeline, constraints, 45_000_000
)
print(result.summary())

In [ ]:
# Which projects were selected?
print("Selected projects:")
for p in result.selected_projects:
    print(f"  {p.project_id}  {p.state}  {p.sector:<22}  ${p.qei_request:,.0f}  {p.expected_jobs_created} jobs")

total_qei = sum(p.qei_request for p in result.selected_projects)
total_jobs = sum(p.expected_jobs_created for p in result.selected_projects)
jpm = total_jobs / (total_qei / 1_000_000)
states_selected = len({p.state for p in result.selected_projects})
print(f"\nTotal QEI: ${total_qei:,.0f}  |  Jobs/$MM: {jpm:.1f}  |  States: {states_selected}")

## 8. Application via the Full `Application.optimize_pipeline()` Interface

In [ ]:
# Same optimizer, accessed directly from the Application object
opt_result = app2.optimize_pipeline(
    constraints=OptimizationConstraints(
        max_total_qei=45_000_000,
        min_states=5,
    )
)

print(f"Alignment before: {opt_result.alignment_score_before * 100:.1f}/100")
print(f"Alignment after:  {opt_result.alignment_score_after * 100:.1f}/100")
print(f"Improvement:      {(opt_result.alignment_score_after - opt_result.alignment_score_before) * 100:+.1f} pts")
print(f"Feasible:         {opt_result.constraints_satisfied}")
print(f"Iterations:       {opt_result.iterations}")
print()
print("Dimensional improvements:")
for dim, delta in opt_result.dimensional_improvements.items():
    arrow = '↑' if delta > 0 else ('↓' if delta < 0 else '—')
    print(f"  {dim.replace('_', ' ').title():<30} {arrow} {delta * 100:+.1f}")

## 9. Methodology Disclosure

Always check the methodology disclosure before interpreting any score:

In [ ]:
print("=" * 70)
print("METHODOLOGY DISCLOSURE")
print("=" * 70)
print(score2.methodology_disclosure)
print()
print("Benchmark methodology:")
print(bc.methodology_disclosure)

## Summary

This notebook demonstrated the full Week 3 intelligence workflow:

| Step | Tool | What You Get |
|---|---|---|
| Benchmark | `app.benchmark()` | 9-metric tier comparison vs. CY2020-2024 winners |
| Alignment Score | `app.score_win_probability()` | 0-100 score across 5 dimensions |
| Recommendations | `app.recommendations()` | Quantified, prioritized improvement actions |
| Pattern Analysis | `compare_to_winners()` | Gap-to-winner-median per dimension |
| Optimization | `app.optimize_pipeline()` | Max-alignment project subset |

**Key reminder:** All scores reflect *alignment with historical NMTC winner patterns*. No true win probability can be computed from public data alone.